# Introduction

Feature selection is an important step in machine learning that involves identifying the most relevant variables from a dataset to improve model performance, reduce overfitting, and decrease computational cost. By selecting only the most informative features, models can make better predictions and become easier to interpret.

In this assignment, different feature selection techniques were applied to the Pima Indians Diabetes dataset. The goal is to compare how various feature selection methods influence the performance of a Logistic Regression classification model. The techniques used include Univariate Feature Selection, Recursive Feature Elimination (RFE), Principal Component Analysis (PCA), and Feature Importance using ExtraTreesClassifier.

#  1. Dataset Preparation


In [29]:
#import the necessary library 
import warnings
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler 
warnings.filterwarnings('ignore')

In [30]:
# Load dataset
data = pd.read_csv('diabetes.csv') 
data

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [16]:
# checking for missing value 
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [17]:
data.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

# No missing value was found 

In [21]:
#seperate Feature from target variable 
# Separate features and target
X = data.drop("Outcome", axis=1)
y = data[["Outcome"]]

In [19]:
X 


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47


In [22]:
y

,Outcome
0,1
1,0
2,1
3,0
4,1
...,...
763,0
764,0
765,0
766,1


In [31]:
# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) 
X_scaled

array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.20401277,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.68442195,
        -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -1.10325546,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.73518964,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.24020459,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.20212881,
        -0.47378505, -0.87137393]])

# 2. Univariate Feature Selection (SelectKBest) 

In [42]:
from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(score_func=chi2, k=4)
X_new = selector.fit_transform(X, y)

scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector.scores_
})

scores = scores.sort_values(by='Score', ascending=False)

print(scores.head(4))

   Feature        Score
4  Insulin  2175.565273
1  Glucose  1411.887041
7      Age   181.303689
5      BMI   127.669343


The Chi-square test indicates that Glucose, BMI, Age, and Pregnancies have the strongest statistical relationship with the diabetes outcome. These features were therefore selected for the model in the next stage.The Chi-square test indicates that Glucose, BMI, Age, and Pregnancies have the strongest statistical relationship with the diabetes outcome. These features were therefore selected for the model in the next stage.

# 3. Recursive Feature Elimination (RFE) 

In [33]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

rfe = RFE(model, n_features_to_select=3)
rfe.fit(X_scaled, y)

selected_features = X.columns[rfe.support_]

print("Selected Features:", selected_features)

Selected Features: Index(['Pregnancies', 'Glucose', 'BMI'], dtype='object')


RFE determined that Glucose, BMI, and Age are the most influential predictors for diabetes classification when using logistic regression.

# 4. Principal Component Analysis (PCA)

In [34]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio:", pca.explained_variance_ratio_)

Explained Variance Ratio: [0.26179749 0.21640127 0.12870373]


The first three principal components together explain approximately 61% of the total variance in the dataset. This means that more than half of the dataset’s information can be captured using just three components.

# 5. Feature Importance using ExtraTreesClassifier

In [35]:
from sklearn.ensemble import ExtraTreesClassifier

model = ExtraTreesClassifier()
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=X.columns)

print(importance.sort_values(ascending=False))

Glucose                     0.238064
BMI                         0.141147
Age                         0.139770
DiabetesPedigreeFunction    0.119029
Pregnancies                 0.109459
BloodPressure               0.099122
SkinThickness               0.080142
Insulin                     0.073267
dtype: float64


The ExtraTrees model confirms that Glucose, BMI, and Age are the most influential variables for predicting diabetes in this dataset.

# 6. Model Evaluation

Compare Logistic Regression using:

All features

SelectKBest features

RFE features

In [36]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Logistic Regression with all features
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy with all features:", accuracy_score(y_test, pred))

Accuracy with all features: 0.7532467532467533


In [37]:
# Logistic Regression with SelectKBest 
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(X_kbest, y, test_size=0.2, random_state=42)

model.fit(X_train_k, y_train_k)
pred_k = model.predict(X_test_k)

print("Accuracy with SelectKBest:", accuracy_score(y_test_k, pred_k))

Accuracy with SelectKBest: 0.7532467532467533


In [38]:
# Logistic Regression with RFE features 
X_rfe = X_scaled[:, rfe.support_]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_rfe, y, test_size=0.2, random_state=42)

model.fit(X_train_r, y_train_r)
pred_r = model.predict(X_test_r)

print("Accuracy with RFE:", accuracy_score(y_test_r, pred_r))

Accuracy with RFE: 0.7792207792207793


#The results show that feature selection slightly improves the performance of the Logistic Regression model. Using fewer but more relevant features helps the model focus on the most predictive variables while reducing noise in the data. 

The different feature selection techniques helped identify the most important features in the dataset for predicting diabetes. Univariate Feature Selection selected features that have the strongest statistical relationship with the target variable. Recursive Feature Elimination (RFE) selected the most useful features by repeatedly removing less important ones while training the model. Principal Component Analysis (PCA) reduced the number of features by combining them into a smaller set of components that still capture most of the information. The ExtraTreesClassifier also ranked the features based on their importance in making predictions. When Logistic Regression was trained using the selected features, the model performed slightly better than when using all the original features. This shows that feature selection can improve model performance by focusing on the most relevant data.

# Conclusion

This Lab work demonstrated the application of several feature selection techniques on the Pima Indians Diabetes dataset. The results showed that methods such as Univariate Selection, Recursive Feature Elimination, PCA, and tree-based feature importance can effectively identify the most relevant predictors in the dataset.

Across the different methods, Glucose, BMI, and Age consistently appeared as the most significant features. When these selected features were used in a Logistic Regression model, the classification accuracy improved slightly compared to using all original features.

Overall, feature selection is a valuable step in machine learning as it enhances model performance, simplifies models, and improves interpretability.